# 04 · Incident retrieval with DuckDB + VSS

The PDR persists YOLOv8 frame embeddings and retrieves *semantically similar*
past incidents. This reproduces that: embed frames, store the 512-d vectors in
DuckDB's VSS extension, and run a cosine-distance similarity query.

In [1]:
import glob, duckdb, numpy as np
from ultralytics import YOLO
DEV = "cuda:0" if __import__("torch").cuda.is_available() else "cpu"
emb_model = YOLO("../backend/yolov8s.pt")
frames = sorted(glob.glob("../tmp/frames/tekno-*.jpg"))
vectors = {f.split('/')[-1].split(chr(92))[-1]: [float(x) for x in emb_model.embed(f, imgsz=512, device=DEV, verbose=False)[0].tolist()] for f in frames}
DIM = len(next(iter(vectors.values())))
print("embedded", len(vectors), "frames | embedding dim:", DIM)

embedded 18 frames | embedding dim: 512


In [2]:
con = duckdb.connect()
con.execute("INSTALL vss"); con.execute("LOAD vss")
con.execute(f"CREATE TABLE incidents(name TEXT, v FLOAT[{DIM}])")
for name, v in vectors.items():
    con.execute("INSERT INTO incidents VALUES (?, ?)", [name, v])
con.execute("CREATE INDEX idx ON incidents USING HNSW(v) WITH (metric='cosine')")
print("stored", con.execute("SELECT COUNT(*) FROM incidents").fetchone()[0], "incident vectors in DuckDB + HNSW index")

stored 18 incident vectors in DuckDB + HNSW index


In [3]:
query = "tekno-01-4s.jpg"
rows = con.execute(f'''
    SELECT name, array_cosine_distance(v, (SELECT v FROM incidents WHERE name=?)::FLOAT[{DIM}]) AS dist
    FROM incidents WHERE name != ? ORDER BY dist ASC LIMIT 5
''', [query, query]).fetchall()
print(f"Most similar incidents to {query}:")
for name, dist in rows:
    print(f"  {name:20} similarity={1-dist:.3f}")

Most similar incidents to tekno-01-4s.jpg:
  tekno-02-4s.jpg      similarity=0.973
  tekno-03-5s.jpg      similarity=0.964
  tekno-02-5s.jpg      similarity=0.956
  tekno-01-5s.jpg      similarity=0.956
  tekno-01-6s.jpg      similarity=0.954


**Conclusion.** YOLOv8 embeddings + DuckDB VSS give real, in-process vector
similarity search over incidents — frames from the same clip/approach rank as most
similar. This backs the live `GET /api/incidents/{id}/similar` endpoint.